In [1]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import os
from cellflow.training import ComputationCallback
from cellflow.preprocessing import transfer_labels, compute_wknn
from cellflow.training import ComputationCallback
from numpy.typing import ArrayLike
from cellflow.metrics import compute_r_squared, compute_e_distance
from cellflow.metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div
import sys
import pickle
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca



/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [5]:
def compute_metrics(adata_ref: ad.AnnData, adata_pred: ad.AnnData, deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)

    deg_r_sq = {}
    for k in deg_genes.keys():
        donor_deg_dict = {k: v for k, v in deg_genes[k].items() if (k.startswith(donor_held_out) and k.endswith(f"_{cytokine}"))}
        deg_r_sq[k] = {}
        for ct_cyto in donor_deg_dict.keys():
            cell_type = ct_cyto.split("_")[1]
            adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
            adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
            if adata_pred_ct.n_obs == 0:
                continue
        
            deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
            deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
            deg_pred_decoded = adata_pred_ct[:,deg_mask].X
            deg_r_sq[k][f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)
        
      
    return deg_r_sq

In [6]:
def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates(subset="cytokine")
    emb_vectors = {}
    for _,row in conds.iterrows():
        cyto = row["cytokine"]
        if cyto=="PBS":
            continue
        emb_vectors[cyto] = embeddings[cyto]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb

In [7]:
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_donor/closest_embedding_different_k"
donor_held_out = "Donor1"
idx_given_donor = "0"

control_key = "is_control"
    
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")
adata_ood_perturbed  = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_ood_{donor_held_out}.h5ad")
cytokines_to_impute = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_impute"]
cytokines_to_train_data = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_train_data"]


adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]


with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs_different_top_k.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

donor_embeddings = adata_train.uns['donor_embeddings']    

adata_ctrl_current_donor = adata_ctrl[adata_ctrl.obs["donor"]==donor_held_out]
if adata_ctrl_current_donor.n_obs > 10000:
        sc.pp.subsample(adata_ctrl_current_donor, n_obs=10000)

closest_donor = find_closest_embedding(donor_embeddings[donor_held_out], {k:v for k,v in donor_embeddings.items() if k!=donor_held_out})
for k in deg_genes.keys():
    for cytokine in cytokines_to_impute:
        adata_pred = adata_train[(adata_train.obs["cytokine"]==cytokine)&(adata_train.obs["donor"]==closest_donor)].copy()
        adata_pred.X = adata_pred.X.toarray()
        project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")

        
        condition = f"{donor_held_out}_{cytokine}"
        
        project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
        project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
        cond_orig = condition
        condition = condition + "_" + str(len(cytokines_to_train_data))
        adata_ood_true = adata_full[(adata_full.obs["donor"] == donor_held_out) & (adata_full.obs["cytokine"]==cytokine)]
        
        out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, deg_dict=deg_genes[k], adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
        df = pd.DataFrame.from_dict(out)
        df["condition"]=condition
        df["num_cytokines_in_train"] = len(cytokines_to_train_data)
        
        df.to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))

        

INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_wknn.py:88: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn


ValueError: cannot use columns parameter with orient='columns'

In [10]:
out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, deg_dict=deg_genes[k], adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
df = pd.DataFrame.from_dict(out)
df["condition"]=condition
df["num_cytokines_in_train"] = len(cytokines_to_train_data)

df.to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))


INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              
INFO     cuML is not installed or GPU is not available. Falling back to neighborhood estimation using CPU with     
         pynndescent.                                                                                              


In [11]:
df

,20,50,100,200,500,1000,condition,num_cytokines_in_train
deg_decoded_r_squared_CD8 Naive,0.153139,0.386068,0.417695,0.600876,0.691783,0.747231,Donor1_OX40L_2,2
deg_decoded_r_squared_B Naive,0.579846,0.661938,0.714752,0.773991,0.842537,0.869210,Donor1_OX40L_2,2
deg_decoded_r_squared_B Intermediate/Memory,0.750355,0.669197,0.672663,0.750174,0.833016,0.860835,Donor1_OX40L_2,2
deg_decoded_r_squared_CD14 Mono,0.654528,0.163495,0.334397,0.587443,0.782221,0.829879,Donor1_OX40L_2,2
deg_decoded_r_squared_CD4 Naive,0.679063,0.727973,0.746631,0.764487,0.806485,0.817047,Donor1_OX40L_2,2
deg_decoded_r_squared_CD56-dim NK,0.527078,0.486736,0.576223,0.703448,0.814323,0.844663,Donor1_OX40L_2,2
deg_decoded_r_squared_CD4 Memory,0.580755,0.529421,0.585058,0.701290,0.786305,0.819065,Donor1_OX40L_2,2
deg_decoded_r_squared_cDC,0.528121,0.593757,0.609874,0.716934,0.807389,0.837907,Donor1_OX40L_2,2
deg_decoded_r_squared_CD8 Memory,0.186507,0.498683,0.434696,0.633837,0.738939,0.786402,Donor1_OX40L_2,2
deg_decoded_r_squared_CD16 Mono,0.624119,0.487059,0.623867,0.788887,0.858339,0.886478,Donor1_OX40L_2,2


In [12]:
len(cytokines_to_train_data)

2

In [19]:
adata_train.uns["split_info"].keys()#["20"]

dict_keys(['0'])

In [22]:
adata_ood_perturbed.uns['cytokines_to_train_data']['64']

array([['IL-34', 'IL-20', 'LIGHT', 'Leptin', 'APRIL', 'RANKL', 'IL-23',
        'IFN-beta', 'IL-17E', 'IL-26', 'TRAIL', 'IL-33', 'IL-17F',
        'IL-8', 'IL-19', 'IL-1-beta', 'TGF-beta1', 'IL-17B', 'IL-17D',
        'IL-4', 'PRL', 'LIF', 'LT-alpha2-beta1', 'TWEAK', 'IL-7',
        'IL-16', 'FLT3L', 'IL-21', 'IL-15', 'OSM', 'IFN-lambda3',
        'GITRL', 'TNF-alpha', 'Decorin', 'IL-2', 'IL-31', 'IL-12',
        'IL-17C', 'IL-17A', 'CD30L', 'C5a', 'IL-3', 'Noggin', 'IL-24',
        'EPO', 'IFN-lambda2', 'IL-1-alpha', 'IFN-alpha1', '4-1BBL',
        'IL-6', 'FGF-beta', 'IL-11', 'IL-36-alpha', 'IGF-1', 'IL-18',
        'TL1A', 'PSPN', 'SCF', 'IL-10', 'IL-9', 'LT-alpha1-beta2',
        'IL-22', 'C3a', 'IL-5', 'PBS'],
       ['APRIL', 'EGF', 'CD40L', 'IL-15', 'IL-8', 'IL-21', 'IL-5',
        'IFN-epsilon', 'IL-17C', 'IL-1-beta', 'IL-18', 'IL-36-alpha',
        'SCF', 'IFN-lambda2', 'RANKL', 'HGF', 'IL-6', 'EPO', 'IL-17B',
        'VEGF', 'TNF-alpha', 'IL-17D', 'IL-23', 'IL-27', 'TGF-beta1

In [23]:
adata_train.uns["split_info"].keys()

dict_keys(['0'])

In [36]:
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(16)}/adata_train_{donor_held_out}.h5ad")

In [37]:
adata_train.uns["split_info"].keys()

dict_keys(['0', '1', '10', '11', '12', '13', '14', '15', '16', '2', '3', '4', '5', '6', '7', '8', '9'])

In [38]:
adata_train.uns["split_info"]['16']

{'cytokines_to_impute': array(['OX40L', 'IL-32-beta', 'IL-1Ra', 'IFN-gamma', 'IFN-omega', 'BAFF',
        'CD27L', 'ADSF', 'FasL', 'M-CSF'], dtype=object),
 'cytokines_to_train_data': array(['APRIL', 'EGF', 'CD40L', 'IL-15', 'IL-8', 'IL-21', 'IL-5',
        'IFN-epsilon', 'IL-17C', 'IL-1-beta', 'IL-18', 'IL-36-alpha',
        'SCF', 'IFN-lambda2', 'RANKL', 'HGF', 'IL-6', 'EPO', 'IL-17B',
        'VEGF', 'TNF-alpha', 'IL-17D', 'IL-23', 'IL-27', 'TGF-beta1',
        'IL-35', 'TPO', 'IL-3', 'IL-2', 'Leptin', 'FLT3L', 'IL-7', 'TWEAK',
        '4-1BBL', 'GITRL', 'TRAIL', 'IFN-lambda3', 'IL-17A', 'LIF',
        'IL-34', 'LT-alpha2-beta1', 'TSLP', 'IL-19', 'Decorin', 'GM-CSF',
        'CT-1', 'PRL', 'IFN-alpha1', 'IL-17F', 'IL-26', 'G-CSF', 'IL-12',
        'LT-alpha1-beta2', 'IL-22', 'TL1A', 'IL-33', 'IL-31', 'C5a',
        'IL-24', 'Noggin', 'IL-1-alpha', 'IL-16', 'IL-4', 'IL-9', 'PBS'],
       dtype=object)}

In [42]:
cytokines_to_train_data = adata_train.uns["split_info"]["16"]["cytokines_to_train_data"]
print(len(cytokines_to_train_data) == 65)
    

True


In [41]:
len(cytokines_to_train_data)

2

In [35]:
for k in adata_train.uns["split_info"].keys():
    print(k, len(adata_train.uns["split_info"][k]["cytokines_to_train_data"]))

0 2
1 2
10 33
11 33
12 5
13 5
14 5
15 65
16 65
17 65
18 9
19 9
2 2
20 9
21 81
22 81
23 81
3 17
4 17
5 17
6 3
7 3
8 3
9 33
